# HalluCheck — FACTS Grounding (860 items) op Gemma-4-31B + dpo_gemma31b_grounding-adapter

**Voor je begint:**
1. Runtime → Wijzig runtime type → GPU → **A100** (of L4 als A100 niet beschikbaar is; T4 is te klein voor dit model).
2. Upload eerst de map `dpo_gemma31b_grounding-adapter` (7 bestanden, ~215MB) naar je Google Drive, bv. naar `MyDrive/hallucheck/dpo_gemma31b_grounding-adapter/`.
3. Upload ook `eval_extract_compose_gemma.py` naar `MyDrive/hallucheck/`.
4. Zorg dat je een Hugging Face account + token hebt, en dat je de licentie van `google/gemma-4-31B-it` (en eventueel `google/FACTS-grounding-public`) hebt geaccepteerd op huggingface.co.

Dit notebook draait de dataset in **chunks** (pas `OFFSET`/`N` per run aan) omdat alle 860 items in één keer te lang duurt voor een enkele Colab-sessie. Resultaten worden weggeschreven naar Google Drive, dus ze blijven staan ook als de runtime later afgesloten wordt.

## 1. Drive mounten + workspace-symlink
Het script schrijft naar hardcoded paden onder `/workspace/...` — deze symlink zorgt dat die paden eigenlijk naar je Drive wijzen, zodat resultaten persistent zijn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/hallucheck_results"
!ln -sf "/content/drive/MyDrive/hallucheck_results" /workspace
!cp "/content/drive/MyDrive/hallucheck/eval_extract_compose_gemma.py" /workspace/
!ls -la /workspace/

## 2. Dependencies
Torch/torchvision/transformers/peft moeten onderling compatibel zijn (dit gaf problemen op RunPod als je alleen `transformers` upgradet zonder torch mee te upgraden) — daarom hier bewust torch/torchvision/torchaudio samen installeren.

In [ ]:
!pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q -U transformers accelerate peft bitsandbytes datasets huggingface_hub
!pip uninstall -y -q hf_xet

import torch, torchvision, transformers, peft
print(torch.__version__, torchvision.__version__, transformers.__version__, peft.__version__, torch.cuda.is_available())

## 3. Hugging Face login
Plak je token in het invoerveld (verschijnt via `getpass`, wordt niet in het notebook-bestand opgeslagen).

In [ ]:
from huggingface_hub import login
from getpass import getpass
login(token=getpass('HF token: '))

## 4. Tokenizer-patch
Dit specifieke checkpoint heeft een bug in `tokenizer_config.json` (`extra_special_tokens` is een lijst i.p.v. dict) die anders een crash geeft bij het laden.

In [ ]:
from huggingface_hub import hf_hub_download
import json

p = hf_hub_download('google/gemma-4-31B-it', 'tokenizer_config.json')
cfg = json.load(open(p))
est = cfg.get('extra_special_tokens')
print('before:', est)
if isinstance(est, list):
    cfg['extra_special_tokens'] = {'video_token': '<|video|>'}
    json.dump(cfg, open(p, 'w'), indent=2)
    print('patched')
else:
    print('geen patch nodig')

## 5. Eén chunk draaien
Pas `OFFSET`, `N` en `LABEL` hieronder aan per sessie/run. Voorstel voor de volledige 860 items in stukken van 150:
- run 1: OFFSET=0,   N=150, LABEL=facts860_colab_p1
- run 2: OFFSET=150, N=150, LABEL=facts860_colab_p2
- run 3: OFFSET=300, N=150, LABEL=facts860_colab_p3
- run 4: OFFSET=450, N=150, LABEL=facts860_colab_p4
- run 5: OFFSET=600, N=150, LABEL=facts860_colab_p5
- run 6: OFFSET=750, N=110, LABEL=facts860_colab_p6

Elke run schrijft naar `MyDrive/hallucheck_results/results_<LABEL>.json`. Zet `DEEPSEEK_KEY` (optioneel, via getpass) als je ook direct de automatische gegrond/ongegrond-score uit het script wil krijgen — anders laat je die regel gewoon weg en beoordeel je de antwoorden later zelf/handmatig, zoals in de rest van dit project.

In [ ]:
import os

OFFSET = 0
N = 150
LABEL = "facts860_colab_p1"

os.environ.update({
    "MODEL": "google/gemma-4-31B-it",
    "ADAPTER_PATH": "/content/drive/MyDrive/hallucheck/dpo_gemma31b_grounding-adapter",
    "USE_4BIT": "1",
    "LABEL": LABEL,
    "IDX_LIST": ",".join(str(i) for i in range(OFFSET, OFFSET + N)),
})

# Optioneel — automatische DeepSeek-grounding-score:
# from getpass import getpass
# os.environ["DEEPSEEK_KEY"] = getpass('DeepSeek API key (optioneel, Enter om over te slaan): ')

!python /workspace/eval_extract_compose_gemma.py

**Herhaal cel 5** met de volgende OFFSET/N/LABEL uit de tabel hierboven — in dezelfde sessie na elkaar als de GPU het volhoudt, anders in een nieuwe Colab-sessie (dan alleen stap 1-4 opnieuw doorlopen, Drive-bestanden blijven staan).

## 6. Alle chunks samenvoegen
Draai dit pas nadat alle gewenste chunks (p1..p6) klaar zijn.

In [ ]:
import json, glob

all_results = []
for fn in sorted(glob.glob("/content/drive/MyDrive/hallucheck_results/results_facts860_colab_p*.json")):
    part = json.load(open(fn))
    print(fn, len(part), "items")
    all_results.extend(part)

all_results.sort(key=lambda r: r["idx"])
out_path = "/content/drive/MyDrive/hallucheck_results/results_facts860_colab_ALL.json"
json.dump(all_results, open(out_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(f"Totaal: {len(all_results)} items -> {out_path}")